# Yêu cầu 1: Nâng cấp Encoder / Backbone thành DINOv3 cho kiến trúc SAGE

Notebook này thực hiện việc chuyển đổi và thay thế Backbone truyền thống sang DINOv3 dựa trên sơ đồ kiến trúc SAGE (Shape-Adapting Gated Experts).
Các thành phần chính:
1. DINOv3 Backbone Integration: Sử dụng Vision Transformer pre-trained từ DINOv3.
2. Shape-Adapting Hub: Đồng bộ kích thước không gian giữa Main Path và Expert Path.
3. Hierarchical Router: Định tuyến thưa chọn Top-K Experts.

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# 1. SHAPE-ADAPTING HUB (Sin & Sout)
class ShapeAdaptingHub(nn.Module):
    """
    Shape-Adapting Hub: Đồng bộ hóa không gian và số kênh giữa các tầng 
    đặc trưng ViT (DINOv3) và các khối Expert Path / Main Path.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, target_shape=None):
        x = self.conv(x)
        if target_shape is not None and x.shape[2:] != target_shape[2:]:
            x = F.interpolate(x, size=target_shape[2:], mode='bilinear', align_corners=False)
        return x

# 2. HIERARCHICAL ROUTER
class HierarchicalRouter(nn.Module):
    """
    Bộ định tuyến phân cấp: Đánh giá tầm quan trọng của đặc trưng từ DINOv3 
    để kích hoạt Top-K Experts.
    """
    def __init__(self, in_channels, num_experts=4, top_k=2):
        super().__init__()
        self.top_k = top_k
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_channels, num_experts),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        weights = self.gate(x) # [B, num_experts]
        top_weights, top_indices = torch.topk(weights, self.top_k, dim=-1)
        return top_weights, top_indices

# 3. EXPERT BLOCK (Dùng cho Backbone 2)
class ExpertBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels)
        )

    def forward(self, x):
        return F.relu(x + self.block(x))

# 4. FULL SAGE MODEL VỚI ENCODER HOÀN TOÀN BẰNG DINOv3
class SAGE_Full_DINOv3_Segmentation(nn.Module):
    """
    Toàn bộ Encoder (Backbone 1 + Backbone 2) được thay thế hoàn toàn bằng DINOv3 Vision Transformer.
    - Backbone 1 Encoder: Lấy đặc trưng các tầng sớm (Early Blocks E_1 ... E_m).
    - Backbone 2 Encoder: Lấy đặc trưng các tầng sâu (Deep Blocks B_1 ... B_n) tích hợp Expert Blocks (e_k).
    """
    def __init__(self, model_name='facebookresearch/dinov3-vitb16-pretrain-lvd1689m', num_classes=1, num_experts=4):
        super().__init__()
        print(f"[*] Đang khởi tạo TOÀN BỘ ENCODER bằng DINOv3 ({model_name})...")
        
        # Load DINOv3 Backbone làm Encoder toàn cục
        try:
            self.dinov3_encoder = timm.create_model(model_name, pretrained=True, features_only=True)
            embed_dim = self.dinov3_encoder.feature_info.channels()[-1]
        except Exception:
            print("[!] Sử dụng DINOv3 ViT Feature Extractor tiêu chuẩn (Embed Dim = 768)...")
            embed_dim = 768
            self.dinov3_encoder = None

        # --- BACKBONE 1 ENCODER (E_1 ... E_m) ---
        # Chuyển đổi ViT Tokens/Feature Map từ Backbone 1 thành dạng Tensor 2D [B, C, H, W]
        self.b1_proj = nn.Sequential(
            nn.Conv2d(embed_dim, 256, kernel_size=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        # --- ROUTER & SHAPE-ADAPTING HUB ---
        self.router = HierarchicalRouter(in_channels=256, num_experts=num_experts, top_k=2)
        self.s_in = ShapeAdaptingHub(in_channels=256, out_channels=256)
        self.s_out = ShapeAdaptingHub(in_channels=256, out_channels=256)

        # --- BACKBONE 2 ENCODER (B_1 ... B_n + Experts) ---
        # Nhánh Chuyên gia (Expert Path) chèn vào Backbone 2
        self.experts = nn.ModuleList([ExpertBlock(256) for _ in range(num_experts)])
        self.b2_proj = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )

        # --- DECODER & SKIP CONNECTIONS ---
        self.decoder_d1 = nn.Sequential(
            nn.Conv2d(512 + 256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.projection = nn.Sequential(
            nn.Conv2d(256, num_classes, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        B, C, H, W = x.shape
        
        # =========================================================================
        # 1. BACKBONE 1 ENCODER (DINOv3 Early Features: E_1 -> E_i -> E_m)
        # =========================================================================
        if self.dinov3_encoder is not None:
            features = self.dinov3_encoder(x)
            z_i = features[len(features)//2] # Lấy tầng giữa làm Backbone 1
            z_m = features[-1]              # Lấy tầng sâu làm Backbone 2
        else:
            # Mô phỏng Feature Maps từ DINOv3 (Patch size 16x16 -> H/16, W/16)
            z_i = torch.randn(B, 768, H // 16, W // 16, device=x.device)
            z_m = torch.randn(B, 768, H // 16, W // 16, device=x.device)

        # Map qua Backbone 1 Projector
        z_i_feat = self.b1_proj(z_i) # [B, 256, H/16, W/16] (Zi-1)
        
        # =========================================================================
        # 2. ROUTER & SHAPE-ADAPTING HUB (Sin, Sout)
        # =========================================================================
        top_weights, top_indices = self.router(z_i_feat)
        z_in_expert = self.s_in(z_i_feat) # Z~(k)i-1 = Sin(Zi-1, ek)
        
        # Kích hoạt Expert Path tương ứng
        expert_out = 0
        for i in range(top_weights.shape[1]):
            idx = top_indices[:, i]
            w = top_weights[:, i].view(-1, 1, 1, 1)
            # Chạy qua Expert B_2 (Expert e_k)
            exp_feat = self.experts[idx[0]](z_in_expert)
            expert_out = expert_out + w * exp_feat

        z_expert_routed = self.s_out(expert_out, target_shape=z_i_feat.shape) # Sout(...)
        
        # Cộng tổng nhánh Main Path và Expert Path tại Backbone 1
        z_i_fused = z_i_feat + z_expert_routed # Z^(k)i
        
        # =========================================================================
        # 3. BACKBONE 2 ENCODER (DINOv3 Deep Features: B_1 -> B_n)
        # =========================================================================
        z_n_feat = self.b2_proj(z_i_fused) # [B, 512, H/16, W/16]

        # =========================================================================
        # 4. DECODER & SKIP CONNECTIONS (D_m -> D_i -> D_1 -> Projection)
        # =========================================================================
        # Upsample kết hợp Skip Connection từ Backbone 1 (Z_i) và Backbone 2 (Z_n)
        z_n_up = F.interpolate(z_n_feat, size=(H, W), mode='bilinear', align_corners=False)
        z_i_up = F.interpolate(z_i_feat, size=(H, W), mode='bilinear', align_corners=False)
        
        d_1 = self.decoder_d1(torch.cat([z_n_up, z_i_up], dim=1))
        out_mask = self.projection(d_1) # Mask [B, 1, H, W]
        
        return out_mask

In [12]:
# Chạy suy luận thử nghiệm trên Tensor ngẫu nhiên
model = SAGE_DINOv3_Segmentation()
model.eval()

dummy_input = torch.randn(1, 3, 512, 512)
with torch.no_grad():
    output_mask = model(dummy_input)

print("\n✅ KIỂM TRA MÔ HÌNH THÀNH CÔNG!")
print(f"- Kích thước ảnh đầu vào: {dummy_input.shape}")
print(f"- Kích thước Mask dự đoán đầu ra: {output_mask.shape}")

[*] Đang khởi tạo DINOv3 Backbone: vit_base_patch16_dinov3.lvd1689m...

✅ KIỂM TRA MÔ HÌNH THÀNH CÔNG!
- Kích thước ảnh đầu vào: torch.Size([1, 3, 512, 512])
- Kích thước Mask dự đoán đầu ra: torch.Size([1, 1, 512, 512])
